In [ ]:
import pandas as pd
import glob
import os
import numpy as np

from classify import classify_file

folder_path = "./pipeline_steps/input_files/raw"

In [ ]:
csv_files = glob.glob(os.path.join(folder_path, "*.csv"))
xlsx_files = glob.glob(os.path.join(folder_path, "*.xlsx"))

all_files = csv_files + xlsx_files

dfs = []

for file in all_files:
    if file.endswith(".csv"):
        df = pd.read_csv(file)
    elif file.endswith(".xlsx"):
        df = pd.read_excel(file)
    else:
        continue
    
    dfs.append(df)

combined_df = pd.concat(dfs, ignore_index=True)

In [ ]:
combined_df

In [ ]:
# your mapping dict (canonical -> list of possible source columns)
COL_MAP = {
    "CaseNumber": ["CaseNum", "CaseNumber", "Case Number", "Case#"],
    "ResidenceType": ["ResType", "ResidenceType", "Residence Type"],
    "DeathDate": ["DeathDate", "Date of Death", "Death Date", "DateOfDeath", "DateofDeath"],
    "DeathTime": ["DeathTime", "Time of Death", "Death Time", "TimeofDeath"],
    "DeathAddress": ["DeathAddress", "DeathAddr", "DeathAdress", "DeathAddr.1", "address.death", "DeathAdress.1", "Death Address"],
    "DeathZip": ["DeathZip", "DeathZip.1", "DeathZipCode", "DeathZi\np", "Zip"],
    "DeathCity": ["DeathCity", "Death City", "DeathCityDesc"],
    "EventPlace": ["EventPlace", "Event Place"],
    "EventAddress": ["EventAddress", "EventAddr", "EventAddr.1", "eventaddress", "Event Address"],
    "EventZip": ["EventZip", "EventZip.1", "EventZi\np", "EventZipCode", "Zip.1", "Event Zip"],
    "EventCity": ["EventCity", "EventCityDesc"],
    "Mode": ["Mode", "Mode.1"],
    "CauseA": ["CauseA", "Cause A", "DeathCauseA"],
    "CauseB": ["CauseB", "Cause B", "DeathCauseB"],
    "CauseC": ["CauseC", "Cause C", "DeathCauseC"],
    "CauseD": ["CauseD", "Cause D", "DeathCauseD"],
    "CauseOther": ["CauseOther", "Other Cause", "OtherCause"],
    "HowInjuryOccurred": ["HowInjuryOccurred", "InjuryDesc", "HowInjuryOccu\nrred"],
    "FirstName": ["First Name"],
    "MiddleName": ["Middle Name"],
    "LastName": ["Last Name"],
    "DateofBirth": ["Date of Birth", "BirthDate"],
    "Text": ["Text", "text"],
    "Address": ["Address", "address"],
    "Race": ["Race", "Races"]
}

def _blank_to_nan(s: pd.Series) -> pd.Series:
    """Treat empty/whitespace strings as missing."""
    if s.dtype == "object":
        s = s.replace(r"^\s*$", np.nan, regex=True)
    return s

def coalesce_columns(df: pd.DataFrame, col_map: dict[str, list[str]], drop_sources: bool = False) -> pd.DataFrame:
    df = df.copy()

    for canon, aliases in col_map.items():
        # keep only aliases that actually exist in df
        present = [c for c in aliases if c in df.columns]
        if not present:
            continue

        # start with an all-missing series
        out = pd.Series(np.nan, index=df.index)

        # fill from each alias in order (first non-missing wins)
        for c in present:
            src = _blank_to_nan(df[c])
            out = out.combine_first(src)

        df[canon] = out

        if drop_sources:
            # don't drop the canonical column if it was also an alias name
            to_drop = [c for c in present if c != canon]
            df = df.drop(columns=to_drop, errors="ignore")

    return df

combined_df = coalesce_columns(combined_df, COL_MAP, drop_sources=True)  # set True if you want to remove old alias cols

In [ ]:
combined_df = combined_df.drop_duplicates(subset='CaseNumber')

In [ ]:
combined_df['DeathDate']

In [ ]:
s = combined_df["DeathDate"]

# Pass 1: parse as-is (won't raise; unparseable rows -> NaT)
death_dt = pd.to_datetime(s, errors="coerce")

# Pass 2: only fix the failures by extracting a date/datetime token
mask = death_dt.isna() & s.notna()

s_bad = s.loc[mask].astype("string")

# Extract first date/datetime-looking token (handles "Facility 5/8/2024", "2022-12-05 00:00:00\tRESIDENCE ...", etc.)
token = s_bad.str.extract(
    r'('
    r'\d{4}-\d{2}-\d{2}(?:[ T]\d{2}:\d{2}:\d{2})?'   # 2022-12-05 or 2022-12-05 00:00:00
    r'|'
    r'\d{1,2}/\d{1,2}/\d{2,4}'                       # 6/1/2023 or 5/8/2024
    r'|'
    r'\d{1,2}-\d{1,2}-\d{2,4}'                       # 6-1-2023
    r')',
    expand=False
)

death_dt.loc[mask] = pd.to_datetime(token, errors="coerce")

combined_df["DeathDate_parsed"] = death_dt

In [ ]:
combined_df['DeathDate'] = combined_df['DeathDate_parsed']

In [ ]:
combined_df['DeathDate'] = pd.to_datetime(combined_df['DeathDate'], format="mixed")

In [ ]:
combined_df = combined_df.sort_values("DeathDate")

In [ ]:
combined_df = combined_df.drop(columns=['DeathDate_parsed'])

In [ ]:
def clean_and_categorize_race(race):
    if pd.isna(race):
        return np.nan
    race = race.lower()  # Case folding
    race = race.replace(" ", "")  # Remove spaces
    race = race.replace("\n", "")  # Remove newline characters
    if '","' in race or "," in race:  # Adjust based on your actual separator
        return "UNKNOWN"

    if race in ["americanindian", "nativeamerican"]:
        return "AMERICAN INDIAN"
    elif race in ["armenian", "middleeastern"]:
        return "MIDDLE EASTERN"
    elif race in [
        "asian",
        "cambodian",
        "chinese",
        "filipino",
        "japanese",
        "korean",
        "eastindian",
        "thai",
        "vietnamese",
    ]:
        return "ASIAN"
    elif race in ["black"]:
        return "BLACK"
    elif race in [
        "guamanian",
        "hawaiian",
        "pacificislander",
        "samoan",
        "tongan",
        "nativehawaiian/otherpacificislander",
    ]:
        return "PACIFIC ISLANDER"
    elif race in ["hispanic/latino", "hispanic/latina", "hispanic/latinamerican"]:
        return "LATINE"
    elif race in ["white", "caucasian", "white/caucasian"]:
        return "WHITE"
    elif race in ["unknown", "null", "unknown/other"]:
        return np.nan
    else:
        return race


In [ ]:
combined_df['Race'] = combined_df['Race'].apply(clean_and_categorize_race)

In [ ]:
MODEL_NAME = "bert_models/bioclinicalbert"

classify_file(combined_df, "classified_all_deaths.csv", MODEL_NAME)